# ILCOF-1 | Configurable 2D workflow

Screen 28 serrated arrangements of ILCOF-1, relax the lowest-energy sampled structure and compare its powder X-ray diffraction (PXRD) pattern with experiment.

- Run from the `configurable/` folder with the supplied fragments and experimental scan.
- Most methods accept optional input and output paths and otherwise use numbered folders under `ILCOF-1/`. Use a separate working directory to compare grids or energy models. [COF-Landscaper API reference](https://cof-landscaper.readthedocs.io/en/stable/classes.html)

In [ ]:
import coflandscaper as cl
from mace.calculators import mace_mp

## Choose the framework and stacking mode

Choose the covalent connectivity within each layer and the stacking arrangements to screen.

- `COF_NAME` identifies the framework in filenames and output folders.
- `TOPOLOGY` supplies the required `topo` argument. Use `"sql"` or `"kgm"` for a four-connected node and a two-connected linker, `"hcb"` for a three-connected node and a two-connected linker, or `"sql_ab"` / `"hcb_ab"` for two complementary four- or three-connected nodes, respectively.
- Set `MODE` once for the stacking scan and subsequent analysis. `"serr"` alternates the lateral slip between layers, `"incl"` repeats it in the same direction and `"both"` screens the two modes separately (API default).

In [ ]:
COF_NAME = "ILCOF-1"
TOPOLOGY = "sql"
MODE = "serr"

## Choose the energy model

Single-layer preoptimisation, single-point screening and final relaxation reuse the same ASE calculator.

- `calculator` accepts an ASE instance, a class or factory, an ASE calculator name or a `"module:attribute"` import path. Pass factory settings through `calculator_kwargs`, e.g. `{"device": "cuda"}`, or configure an instance directly as below. [ASE calculator catalogue](https://docs.ase-lib.org/ase/calculators/calculators.html)
- `model="mh-1"` selects the checkpoint and `head="spice_wB97M"` selects its energy reference, matching the COF-Landscaper defaults. Other MH-1 heads include `omol`, `omat_pbe`, `matpes_r2scan`, `mp_pbe_refit_add` and `oc20_usemppbe`. [MACE-MH-1 model card](https://huggingface.co/mace-foundations/mace-mh-1)
- `dispersion=False` omits an extra D3 correction for `spice_wB97M`. The package preset enables D3 for `omat_pbe` and `matpes_r2scan`. Set the corresponding D3 parameters explicitly when using either head with `mace_mp`.
- `device` selects CPU or a supported GPU (`"cuda"`). `default_dtype` selects `"float64"` or `"float32"` precision. The package defaults are `"cpu"` and `"float64"`.
- A replacement calculator must supply energy for screening and forces and stress for cell relaxation.

In [ ]:
calculator = mace_mp(model="mh-1", head="spice_wB97M", device="cpu", default_dtype="float64", dispersion=False)

### Optional UMA calculator

With `fairchem-core` and access to the UMA checkpoint, you can replace the MACE cell with this calculator for the `omat` bulk-materials task. [FAIR Chemistry example](https://facebookresearch.github.io/fairchem/formation-energy/)

```python
from fairchem.core.calculate import FAIRChemCalculator

calculator = FAIRChemCalculator.from_model_checkpoint("uma-s-1p2p1", task_name="omat")
```

## Build the single layer

Assemble the pyrene-derived node and phenylenediamine-derived linker into an SQL layer using the He markers as connection sites. [PORMAKE building blocks and topology examples](https://github.com/Sangwon91/PORMAKE)

- `input_nodes` and `input_linkers` default to the XYZ files in `0_node/` and `0_linker/`. To choose specific fragments, pass lists such as `input_nodes=["0_node/ILCOF-1_node.xyz"]`. For a direct node–node net use two node files and `input_linkers=[]`.

In [ ]:
cl.BuildCOF2D().build(TOPOLOGY, COF_NAME)

## Preoptimise the isolated layer

Relax the single sheet before stacking so its in-plane dimensions reflect the chosen energy model.

- `topo=TOPOLOGY` fixes the ideal angles (90°, 90°, 90° for SQL) and the cell height (15 Å) while allowing *a* and *b* to relax independently. Omitting `topo` allows the full cell to relax.
- `fix_z=True` (preoptimisation default) fixes atomic Cartesian *z* coordinates. `False` allows out-of-plane motion.
- `fmax` sets the force threshold in eV/Å. Preoptimisation defaults to 0.03 unless set on `GeometryOpt`. A value on `run_preopt` overrides it for this stage. [ASE structure optimisation](https://docs.ase-lib.org/ase/optimize.html)
- `max_steps` limits each relaxation to 2,000 steps by default. Unconverged preoptimisation saves the intermediate CIF and raises an error.

In [ ]:
cl.GeometryOpt(calculator=calculator, max_steps=2000).run_preopt(
    COF_NAME, topo=TOPOLOGY, fmax=0.03, fix_z=True
)

## Sample layer separation and slip

Four interlayer distances (ILDs) and seven interlayer slips (ILSs) give 28 serrated configurations along the first lattice vector.

- `ild_start`, `ild_end` and `ild_step` set the separation range and increment in Å (3.2–3.8 by 0.2 here, 3.0–4.0 by 0.1 by default).
- `ils_length_start`, `ils_length_end` and `ils_length_step` set the slip range and increment in Å (0–6 by 1 here). Defaults are 0, the topology-derived endpoint (`None`) and 1, respectively.
- `slip_path` selects a half-cell vector for the square nets from `"a/2"`, `"b/2"` and `"(a+b)/2"`. Omitting it uses the diagonal for `sql` and `a/2` for `sql_ab`. This example explicitly scans along `a/2`.
- `ils_angle` overrides the in-plane direction in degrees from Cartesian *x*, e.g. `45.0`. The default `None` uses the direction of the selected path.
- `print_shift=True` reports the automatically calculated shift length and angle when either is needed (default `False`).

In [ ]:
cl.CreateMatrix(
    ild_start=3.2,
    ild_end=3.8,
    ild_step=0.2,
    ils_length_start=0.0,
    ils_length_end=6.0,
    ils_length_step=1.0,
    slip_path="a/2",
).run(COF_NAME, topo=TOPOLOGY, mode=MODE)

## Calculate single-point energies

Evaluate each sampled stack at fixed coordinates and cell dimensions to rank configurations within each stacking mode.

In [ ]:
cl.SinglePoint(calculator=calculator).run(COF_NAME, mode=MODE)

## Plot the stacking energy landscape

Locate low-energy regions by plotting each sampled energy relative to the lowest energy in its stacking mode.

- `show=True` also displays the saved PNG in the notebook (default `False`).
- `colorscheme` selects a Matplotlib colormap (default `"viridis"`). `rel_energy_max` caps plotted energies in eV, e.g. `0.5` (default `None`, no cap). [Matplotlib colormap gallery](https://matplotlib.org/stable/gallery/color/colormap_reference.html)
- `show_minima_markers` defaults to `True`. The default `minima_mode="global"` marks the grid minimum, while `"local"` also marks detected local minima.
- `show_title_block=True` adds the title and stacking mode (default `False`). `show_header=False` suppresses that block even if enabled (default `True`).
- `method_label` adds an optional energy-model label to the title block, e.g. `"MACE-MH-1"` (default `None`).
- `dft=False` uses the ASE workflow results by default. `True` selects the DFT results in methods that offer this switch.

In [ ]:
cl.Landscape().run(COF_NAME, mode=MODE, show=True)

## Select starting structures for relaxation

Choose sampled configurations to relax, either from their energy ranking or from a specific layer separation and slip.

- `autoselect_minima="global"` selects the lowest sampled energy per mode (default). `"local"` selects detected local minima.
- `selections_serr` and `selections_incl` accept lists of sampled `(ILD, ILS)` pairs in Å, e.g. `[(3.4, 1.0), (3.6, 2.0)]` (default `None`).
- `include_autoselect=True` combines automatic and manual choices (default). `False` uses only the specified pairs.
- `energy_csv` overrides the default energy table for a single mode, e.g. `energy_csv="alternative_energies.csv"`.

In [ ]:
cl.SelectCofs().run(COF_NAME, mode=MODE, autoselect_minima="global")

## Relax the selected stack

Relax the atoms and the full stacked cell so that the layer separation and slip can move away from their starting grid point.

- `GeometryOpt(fmax=0.01)` uses the tighter final force threshold in eV/Å (default). The constructor’s `fix_z=False` default leaves atoms free to relax in all three directions.
- `save_opt_energies_csv=True` writes per-layer energies and the `stopped_due_to_max_steps` flag (default). `False` skips this table.

In [ ]:
cl.GeometryOpt(calculator=calculator, fmax=0.01, max_steps=2000).run(
    COF_NAME, mode=MODE
)

## Measure the relaxed geometry

Measure the final ILD and ILS in Å and compare them with the starting grid values.

- `source="opt"` reads the fully optimised structures (default). `"postopt"` reads fixed-cell structures from the optional scaling workflow.
- `print_values=None` lets Jupyter display the returned table and prints values in scripts. `True` or `False` controls console printing explicitly.
- `final_structures.csv` retains full-precision geometry and includes per-layer energies when the optimisation energy table is available.

In [ ]:
cl.AnalyzeStacking().analyze(COF_NAME, mode=MODE)

## Simulate powder diffraction

Calculate the Bragg reflections of the relaxed periodic structure at the experimental X-ray wavelength.

- `wavelength="CuKa1"` selects Cu Kα₁ (approximately 1.5406 Å) for this scan. The API default is `"CuKa"`. [pymatgen wavelength presets](https://pymatgen.org/pymatgen.analysis.diffraction.html#pymatgen.analysis.diffraction.xrd.XRDCalculator.AVAILABLE_RADIATION)
- `two_theta_range` sets the simulated angular range in degrees 2θ, e.g. `(1.5, 30.0)` (default).
- `export_hkl=True` writes Miller indices alongside the simulated XY profile (default `False`). `max_reflections` sets how many of the strongest reflections to export (default 10).

In [ ]:
cl.PXRD(wavelength="CuKa1").run(COF_NAME, mode=MODE, export_hkl=True)

## Compare simulated and experimental PXRD

Compare peak positions and relative intensities from the relaxed stack with the measured pattern over 2–30° in 2θ. [ILCOF-1 experimental data source](../../../DATA_SOURCES.md#ilcof-1)

In [ ]:
cl.PXRD().plot_sim_vs_exp(COF_NAME, mode=MODE, xlim=(2, 30), show=True)

## Optional · Compare all sampled stacks by PXRD

Compare the experimental pattern with every unrelaxed configuration to see how diffraction changes across the stacking grid.

- `exp_xy_file` selects the measured trace if needed, e.g. `"experimental_pxrd/ILCOF-1.xy"`. By default the folder must contain exactly one XY file.
- `export_xy=True` also saves the individual simulated profiles. The default `False` produces one multipage PDF per mode.

In [ ]:
cl.PXRD(wavelength="CuKa1").screen_landscape(COF_NAME, mode=MODE)